# Final evaluation: one LoRA adapter on What'sUp, VSR and POPE

Runtime → Change runtime type → **T4 GPU**. Set `RUN` in the first cell, one account per run.

Uses the same `evaluation_inputs.zip` (code and benchmark data) as the milestone evaluation, so the
base-model results in `results/evaluation_milestone_vision_fp32/` are reused and the base model is
not rerun. The adapter is read from `spatial_probe/runs/<RUN>/adapter`. Results go to
`spatial_probe/results/evaluation_final/` as `<RUN>_<benchmark>`. About 1 hour per run.

If Colab disconnects, run all cells again: finished benchmarks are skipped, an unfinished one restarts.

In [ ]:
RUN = 'lora_targeted'   # or 'lora_all', 'lora_random'

from google.colab import drive
from pathlib import Path
import hashlib, importlib.metadata, json, subprocess, sys, tempfile, time, zipfile

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/spatial_probe')
BUNDLE = DRIVE_ROOT / 'evaluation_inputs.zip'
ADAPTER = DRIVE_ROOT / 'runs' / RUN / 'adapter'
OUT = DRIVE_ROOT / 'results/evaluation_final'
# Bundle the base runs used (results/evaluation_milestone_vision_fp32/evaluation_inputs.sha256).
MILESTONE_BUNDLE_SHA256 = '05f6dc3046c8381fa4e3941ef1aba01ecd91e11a5357c73ad521f7388f945f58'

assert RUN in ('lora_targeted', 'lora_all', 'lora_random'), RUN
assert (ADAPTER / 'adapter_model.safetensors').is_file(), f'no finished adapter at {ADAPTER}'
print(RUN, 'LoRA modules:', json.loads((ADAPTER / 'adapter_config.json').read_text())['target_modules'])

In [ ]:
# Same code as the base runs, unpacked to local disk so images are not read from Drive.
with BUNDLE.open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == MILESTONE_BUNDLE_SHA256, 'not the bundle the base runs used'
REPO = Path(tempfile.mkdtemp(prefix='spatial-evaluation-', dir='/content'))
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(REPO)
DATA = REPO / 'data'
print('Unpacked to', REPO)

In [ ]:
# Same environment as the milestone notebook: a venv over Colab's CUDA torch, transformers/peft pinned.
gpu = json.loads(subprocess.check_output([sys.executable, '-c', '''
import json, torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
p = torch.cuda.get_device_properties(0)
print(json.dumps({'name': p.name, 'vram_gib': round(p.total_memory / 1024**3, 2),
                  'torch': torch.__version__, 'cuda': torch.version.cuda}))
'''], text=True))
print(gpu)
cuda_versions = {name: importlib.metadata.version(name) for name in ('torch', 'torchvision')}
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(REPO / '.venv')], check=True)
PYTHON = REPO / '.venv/bin/python'
constraints = REPO / '.venv/colab-cuda-constraints.txt'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in cuda_versions.items())
                       + 'transformers==5.22.1\npeft==0.21.2\n')
names = {'transformers', 'accelerate', 'peft', 'numpy', 'pillow', 'pyyaml'}
requirements = [line.strip() for line in (REPO / 'requirements.txt').read_text().splitlines()
                if any(line.lower().startswith(name + '>=') for name in names)]
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements], check=True)

def run(*args):
    print('Running:', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(PYTHON), '-u', *map(str, args)], cwd=REPO, check=True)

OUT.mkdir(parents=True, exist_ok=True)
snapshot = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(OUT / f"environment_{RUN}_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.txt").write_text(
    f'Code/input bundle SHA-256: {MILESTONE_BUNDLE_SHA256}\nAdapter: {ADAPTER}\nGPU: {json.dumps(gpu)}\n\n' + snapshot)

In [ ]:
BENCHMARKS = {
    'whatsup': ['--dataset', 'whatsup', '--data-root', DATA],
    'vsr_zeroshot_test': ['--dataset', 'vsr', '--annotations', DATA / 'vsr/zeroshot/test.jsonl', '--image-root', DATA / 'coco'],
    'pope_random': ['--dataset', 'pope', '--annotations', DATA / 'pope/coco_pope_random.json', '--image-root', DATA / 'coco'],
}
for name, args in BENCHMARKS.items():
    if (OUT / 'metrics' / f'{RUN}_{name}.json').exists():
        print(name, 'already done')
        continue
    run('-m', 'src.eval', *args, '--adapter', ADAPTER, '--model-name', RUN, '--out', OUT)

for name in BENCHMARKS:
    m = json.loads((OUT / 'metrics' / f'{RUN}_{name}.json').read_text())
    print(name, 'accuracy', round(m['accuracy'], 4), {k: round(m[k], 4) for k in ('lateral_accuracy', 'depth_accuracy') if k in m})